In [15]:
import sys
sys.path.append("/var/www/python/Prod/nighthawk/")
import pandas as pd
import numpy as np

from nighthawk.data.pipeline.common_functions import wind, load, genoutage
from nighthawk.util import dataframe_functions, bigquery_functions, connections, sql_functions
from nighthawk.models.portfolioOpt import ve_portfolio_constructor
from nighthawk.data.network import node
import gc
pd.set_option('display.max_columns', None)



In [16]:
# start_dt = '2022-01-11' # no Darwin data from 2022-01-01 to 2022-01-10 
start_dt = '2020-01-01'
end_dt = '2026-05-19'
opexchange = 'SPP'
strategy_list_dict = {'SPP': ['Darwin', 'Fourier']}
file_loc = '/mnt/disks/filedisk2a/Jessica/tmp/'

ve_port = ve_portfolio_constructor.VEPortfolioConstructor(opexchange)


## Fetch Data

In [17]:
metric_cols = ['bid_mw', 'clear_mw', 'profit', 'profit_congestion', 'profit_slack',
       'profit_oprate', 'profit_pos',
       'profit_neg', 'investment']


def calculate_derived_profit_cols(df, da_total_col = 'da_total', min_mw = 0.1, bid_mw_col = 'bid_mw', bid_price_col = 'bid_price'):
    df = df[df['bid_mw'] > min_mw].copy()
    
    df['clear_mw'] = np.where(df['incdec'] == 'Decrement', df['bid_mw'] * (df['bid_price'] > df[da_total_col]),
                              df['bid_mw'] * (df['bid_price'] < df[da_total_col])).round(1)
    df['profit'] = np.where(df['incdec'] == 'Decrement', df['clear_mw'] *
                            (df['rt_total'] - df[da_total_col] -
                             df['op_rate_dec_a']),
                            df['clear_mw'] *
                            (df[da_total_col] - df['rt_total'] - df['op_rate_inc_a']))
    df['profit_congestion'] = np.where(df['incdec'] == 'Decrement',
                                       df['clear_mw'] *
                                       (df['rt_congestion'] -
                                        df['da_congestion']),
                                       df['clear_mw'] *
                                       (df['da_congestion'] - df['rt_congestion']))
    df['profit_slack'] = np.where(df['incdec'] == 'Decrement',
                                  df['clear_mw'] *
                                  (df['rt_slack'] - df['da_slack']),
                                  df['clear_mw'] *
                                  (df['da_slack'] - df['rt_slack']))
    df['profit_oprate'] = np.where(df['incdec'] == 'Decrement', -df['clear_mw'] * df['op_rate_dec_a'],
                                   -df['clear_mw'] * df['op_rate_inc_a'])

    if opexchange.upper() == 'SPP':  # collateral rule for SPP
        df['bidCollateral'] = np.where(df['incdec'] == 'Decrement', abs(df['bid_mw'] * df['bid_ref_price']),
                                       abs(df['bid_mw'] * df['offer_ref_price']))
        df['holdCollateral'] = np.where(df['incdec'] == 'Decrement', abs(df['clear_mw'] * df['bid_ref_price']),
                                        abs(df['clear_mw'] * df['offer_ref_price']))
    else:
        df['bidCollateral'] = np.where(df['incdec'] == 'Decrement', df['bid_mw'] * df['bid_ref_price'],
                                       df['bid_mw'] * df['offer_ref_price'])
        df['holdCollateral'] = np.where(df['incdec'] == 'Decrement', df['clear_mw'] * df['bid_ref_price'],
                                        df['clear_mw'] * df['offer_ref_price'])

    df['profit_pos'] = np.maximum(df['profit'], 0)
    df['profit_neg'] = np.minimum(df['profit'], 0)
    df['investment'] = df['clear_mw'] * df[da_total_col]
    df['investment'] = np.where(df['incdec'] == 'Increment', df['investment'], -df['investment'])
    return df



def generateSummaryStatistics(inputDataFrame):
    numRows = inputDataFrame.shape[0]
    if numRows > 10:
        #         print("Calculating Summary")
        count = inputDataFrame[['dt', 'hr']].drop_duplicates().shape[0]
        totalDt = inputDataFrame['dt'].drop_duplicates().shape[0]
        totalBidMw = inputDataFrame['bid_mw'].sum()
        totalClearedMw = inputDataFrame['clear_mw'].sum()
        clearingPercentage = totalClearedMw*100/totalBidMw

        totalopRate = -1*inputDataFrame['profit_oprate'].sum()
        netProfit = inputDataFrame['profit'].sum()
        positiveHourlyProfit = inputDataFrame['profit_pos'].sum()
        negativeHourlyProfit = inputDataFrame['profit_neg'].sum()
        hourlyTradeRatio = -1*positiveHourlyProfit/negativeHourlyProfit
        profitPerBidMw = netProfit/totalBidMw
        profitPerClearedMw = netProfit/totalClearedMw
        congestionTotalProfit = inputDataFrame['profit_congestion'].sum()
        congestionPositiveProfit = inputDataFrame[inputDataFrame['profit_congestion'] > 0].profit_congestion.sum(
        )
        congestionNegativeProfit = inputDataFrame[inputDataFrame['profit_congestion'] < 0].profit_congestion.sum(
        )
        congestionHourlyTradeRatio = -1*congestionPositiveProfit/congestionNegativeProfit

        averageHourlyProfit = inputDataFrame['profit'].mean()
        # added this for SPP
        var1Percentile = np.percentile(inputDataFrame['profit'].values, 1)
        var5Percentile = np.percentile(inputDataFrame['profit'].values, 5)
        ROVar1Percent = -1*averageHourlyProfit/var1Percentile
        ROVar5Percent = -1*averageHourlyProfit/var5Percentile
        averageHourlyBidMw = inputDataFrame['bid_mw'].mean()
        averageHourlyClearedMw = inputDataFrame['clear_mw'].mean()
        emptyDataFrame = pd.DataFrame({'hrCount': [count],
                                       'totalDt': [totalDt],
                                       'totalBidMw': [round(totalBidMw, 2)],
                                       'clearingPercent': [round(clearingPercentage, 2)],
                                       'netProfit': [round(netProfit)],
                                       'hourlyTR': [round(hourlyTradeRatio, 2)],
                                       'profitPerBidMw': [round(profitPerBidMw, 2)],
                                       'profitPerClearedMw': [round(profitPerClearedMw, 2)],
                                       'congestionTotalProfit': [congestionTotalProfit],
                                       'congestionHourlyTradeRatio': [round(congestionHourlyTradeRatio, 2)],

                                       'averageHourlyProfit': [averageHourlyProfit],
                                       'var1Percentile': [var1Percentile],
                                       'var5Percentile': [var5Percentile],
                                       'ROVar1Percent': [ROVar1Percent],
                                       'ROVar5Percent': [ROVar5Percent],

                                       'totalBidMw': [totalBidMw],
                                       'totalClearedMw': [totalClearedMw],
                                       'averageHourlyBidMw': [averageHourlyBidMw],
                                       'averageHourlyClearedMw': [averageHourlyClearedMw],

                                       })
    else:
        emptyDataFrame = pd.DataFrame()

    import gc
    gc.collect()
    return (emptyDataFrame)

In [18]:

def get_gas_price(start_dt, end_dt):
#     henryHub, tetcoM3Hub, consumersHub, transcoHub, panHandleHub
    dthr = dataframe_functions.create_dthr_df_from_date_range(start_dt, end_dt)    

    gas_forecast = sql_functions.download_df_from_sql_db(f""" SELECT dt, henryHub as natural_gas_henry_f, 
    panHandleHub as natural_gas_panhandle_f FROM 
            odessa_VirtualStrategy_MISO.iceVariables where dt BETWEEN '{start_dt}' and '{end_dt}'
            """)
    gas_forecast['dt'] = gas_forecast['dt'].astype(str)
    gas_forecast['natural_gas_henry_f'] = gas_forecast['natural_gas_henry_f'].replace(0, np.nan)
    gas_forecast['natural_gas_panhandle_f'] = gas_forecast['natural_gas_panhandle_f'].replace(0, np.nan)
    gas_forecast['natural_gas_mix_f'] = gas_forecast[['natural_gas_henry_f', 'natural_gas_panhandle_f']].max(axis=1)
#     gas_forecast['natural_gas_mix_f'] = np.where(gas_forecast['natural_gas_panhandle_f'].isna(), 
#                                                  gas_forecast['natural_gas_henry_f'], gas_forecast['natural_gas_panhandle_f'])
    gas_forecast = pd.merge(dthr[['dt']].drop_duplicates(), gas_forecast, on=['dt'], how='left')
    gas_forecast = gas_forecast.set_index('dt').interpolate(method='linear', axis=0).bfill().ffill().reset_index()

    return gas_forecast

# daily gas price
henry_start_dt = '2018-09-26'
panhandle_start_dt = '2019-03-09'
gas_forecast = get_gas_price(start_dt, end_dt)
gas_forecast['natural_gas_henry_f'] = np.where(gas_forecast['dt'] < henry_start_dt, np.nan, gas_forecast['natural_gas_henry_f'])
gas_forecast['natural_gas_mix_f'] = np.where(gas_forecast['dt'] < henry_start_dt, np.nan, gas_forecast['natural_gas_mix_f'])
gas_forecast['natural_gas_panhandle_f'] = np.where(gas_forecast['dt'] < panhandle_start_dt, np.nan, gas_forecast['natural_gas_panhandle_f'])


# q_lst = [0, 0.005, 0.01, 0.1, 0.4, 0.6, 0.9, 0.99, 0.995, 1]
q_lst = [0, 2, 3, 4.5, 6.5, 9, 100]

gas_cols = [col for col in gas_forecast.columns if col not in ['dt']]
for col in gas_cols:
#     gas_forecast[col + '_bkt'] = pd.qcut(gas_forecast[col].round(1), q_lst, duplicates='drop')
    gas_forecast[col + '_bkt'] = pd.cut(gas_forecast[col].round(1), q_lst, duplicates='drop')
#     gas_forecast[col + '_bkt_label'] = pd.qcut(gas_forecast[col].round(1), q_lst, labels=False, duplicates='drop') + 1
    gas_forecast[col + '_bkt_label'] = pd.cut(gas_forecast[col].round(1), q_lst, labels=False, duplicates='drop') + 1

print(gas_forecast.columns)
display(gas_forecast.head())



Index(['dt', 'natural_gas_henry_f', 'natural_gas_panhandle_f',
       'natural_gas_mix_f', 'natural_gas_henry_f_bkt',
       'natural_gas_henry_f_bkt_label', 'natural_gas_panhandle_f_bkt',
       'natural_gas_panhandle_f_bkt_label', 'natural_gas_mix_f_bkt',
       'natural_gas_mix_f_bkt_label'],
      dtype='object')


,dt,natural_gas_henry_f,natural_gas_panhandle_f,natural_gas_mix_f,natural_gas_henry_f_bkt,natural_gas_henry_f_bkt_label,natural_gas_panhandle_f_bkt,natural_gas_panhandle_f_bkt_label,natural_gas_mix_f_bkt,natural_gas_mix_f_bkt_label
0,2020-01-01,2.05,1.66,2.05,"(0.0, 2.0]",1,"(0.0, 2.0]",1.0,"(0.0, 2.0]",1.0
1,2020-01-02,2.05,1.62,2.05,"(0.0, 2.0]",1,"(0.0, 2.0]",1.0,"(0.0, 2.0]",1.0
2,2020-01-03,2.05,1.58,2.05,"(0.0, 2.0]",1,"(0.0, 2.0]",1.0,"(0.0, 2.0]",1.0
3,2020-01-04,2.05,1.54,2.05,"(0.0, 2.0]",1,"(0.0, 2.0]",1.0,"(0.0, 2.0]",1.0
4,2020-01-05,2.05,1.54,2.05,"(0.0, 2.0]",1,"(0.0, 2.0]",1.0,"(0.0, 2.0]",1.0


In [19]:
def generate_variable_bucket_stats(df): 
    variable_bucket_pct = {}
    variable_cols = [col for col in df.columns if '_bkt_label' in col]
    for variable_name in variable_cols: 
        variable_name = variable_name.replace('_bkt_label', '')
        temp = df[[variable_name+'_bkt', variable_name+'_bkt_label']].copy()
        temp[variable_name+'_cnt'] = 1
        temp = temp.groupby([variable_name+'_bkt', variable_name+'_bkt_label'])[variable_name+'_cnt'].sum().reset_index()
        temp = temp[temp[variable_name+'_cnt']!=0]
#         temp.columns = [variable_name+'_bkt', variable_name+'_bkt_label', variable_name+'_cnt']
        temp[variable_name+'_pct'] = temp[variable_name+'_cnt']/temp[variable_name+'_cnt'].sum()
        temp[variable_name+'_pct'] = temp[variable_name+'_pct'].cumsum().round(3)
        variable_bucket_pct[variable_name] = temp
        
    return variable_bucket_pct

gas_forecast_stats = generate_variable_bucket_stats(gas_forecast)


In [20]:
# portfolio
portfolio = pd.read_csv(file_loc + 'portfolio_df_May2026.csv')
portfolio['dt'] = portfolio['dt'].astype(str)
portfolio = portfolio.merge(gas_forecast[['dt', 'natural_gas_henry_f']])
portfolio['bid_price'] = np.where((portfolio['strategy'].isin(['Darwin', 'DarwinHighCap'])) &(portfolio['incdec'] =='Decrement')&(portfolio['natural_gas_henry_f'] <= 4.5)&(portfolio['bid_price'] >= 150), 150, portfolio['bid_price'])
portfolio['bid_price'] = np.where((portfolio['strategy'].isin(['Darwin', 'DarwinHighCap'])) &(portfolio['incdec'] =='Decrement') &(portfolio['bid_price'] >= 250), 250, portfolio['bid_price'])
portfolio['bid_price'] = np.where((portfolio['strategy'].isin(['Darwin', 'DarwinHighCap'])) &(portfolio['incdec'] =='Increment') &(portfolio['natural_gas_henry_f'] >= 4.5)&(portfolio['bid_price'] < -50), -50, portfolio['bid_price'])
portfolio['bid_price'] = np.where((portfolio['strategy'].isin(['Darwin', 'DarwinHighCap'])) &(portfolio['incdec'] =='Increment') &(portfolio['bid_price'] <= -100), -100, portfolio['bid_price'])
portfolio['bid_price'] = np.where((portfolio['strategy'].isin(['Eigen'])) &(portfolio['incdec'] =='Decrement') &(portfolio['natural_gas_henry_f'] <= 4.5)&(portfolio['bid_price'] >= 200), 200, portfolio['bid_price'])
portfolio['bid_price'] = np.where((portfolio['strategy'].isin(['Eigen'])) &(portfolio['incdec'] =='Decrement') &(portfolio['bid_price'] >= 300), 300, portfolio['bid_price'])
portfolio['bid_price'] = np.where((portfolio['strategy'].isin(['Eigen'])) &(portfolio['incdec'] =='Increment') &(portfolio['natural_gas_henry_f'] >= 6)&(portfolio['bid_price'] < -50), -50, portfolio['bid_price'])
portfolio['bid_price'] = np.where((portfolio['strategy'].isin(['Eigen'])) &(portfolio['incdec'] =='Increment') &(portfolio['bid_price'] <= -100), -100, portfolio['bid_price'])
portfolio = portfolio.drop(columns=['natural_gas_henry_f'])

target_scale_factor = {'Darwin': 0.9, 'Eigen': 0.25, 'Fourier': 0.65, 'DarwinHighCap': 0.9}
portfolio['scale_factor'] = portfolio['strategy'].map(target_scale_factor)
portfolio['bid_mw'] = portfolio['bid_mw']*portfolio['scale_factor']
portfolio = portfolio[portfolio['bid_mw'] > 0].drop(columns=['scale_factor'])


In [13]:

# holdout
holdout_mon_lst = ['2020-03', '2020-06', '2020-10',
    '2021-02', '2021-05', '2021-09', 
    '2022-01',  '2022-07' '2022-12', 
    '2023-04', '2023-08', '2023-11',
    '2024-01', '2024-09', '2024-12', 
    '2025-03', '2025-07']
all_dt_df = pd.DataFrame.from_dict({'dt': pd.date_range(start_dt, end_dt)})
all_dt_df['yr'] = all_dt_df['dt'].dt.year
all_dt_df['mon'] = all_dt_df['dt'].dt.month
all_dt_df['yr_mon'] = all_dt_df['dt'].dt.strftime("%Y-%m")
all_dt_df['dt'] = all_dt_df['dt'].dt.strftime("%Y-%m-%d")
all_dt_df['holdout'] = np.where(all_dt_df['yr_mon'].isin(holdout_mon_lst), 1, 0)
    

ref_df = pd.read_csv(file_loc + 'reference_df.csv')
ref_df['dt'] = ref_df['dt'].astype(str)
ref_df['hr'] = ref_df['hr'].astype(int)
ref_df['node_num'] = ref_df['node_num'].astype(int)

# go_nogo_df
go_nogo_df = pd.read_csv(file_loc + 'go_nogo_days.csv')
go_nogo_df['dt'] = go_nogo_df['dt'].astype(str)
go_nogo_df['hr'] = go_nogo_df['hr'].astype(int)
print(go_nogo_df['nogo'].unique())

tier1_dt = set(go_nogo_df.loc[go_nogo_df['nogo'] == 1, 'dt'].values)
tier2_dt = set(go_nogo_df.loc[go_nogo_df['nogo'] == 2, 'dt'].values)


[0 2 1]


In [ ]:

portfolio_normal_df = portfolio.merge(go_nogo_df.loc[go_nogo_df['nogo'] == 0, ['dt', 'hr']])
portfolio_normal_df['bid_mw_bfr'] = portfolio_normal_df['bid_mw'].copy()

profit_normal_df = calculate_derived_profit_cols(portfolio_normal_df.merge(ref_df))

profit_normal_df['direction'] = np.where((profit_normal_df['rt_congestion'] - profit_normal_df['da_congestion'])*profit_normal_df['da_congestion'] < 0, 'short', 'long')

print(profit_normal_df.columns)
print(profit_normal_df['strategy'].unique())

In [ ]:
# no cut
portfolio_normal_df['bid_mw'] = portfolio_normal_df['bid_mw_bfr'].copy()
_, _, temp = ve_port.get_portfolio_performance(portfolio_normal_df, get_lmp_and_oprate=False)
display(temp)

# no cut from 2023
_, _, temp = ve_port.get_portfolio_performance(portfolio_normal_df[portfolio_normal_df['dt'] >= '2023-01-01'], get_lmp_and_oprate=False)
display(temp)

## SPP Constraint Family Exposure Risk Cut

In [ ]:
import re
from nighthawk.data.pipeline.var_handler import ice_elec_price_vh
from nighthawk.data.product.ve import DailyBidsManager
from nighthawk.data.pipeline.constraint_family_pipeline import PortfolioExposureManager

# set the bid date to test; change to loop over multiple dates for backtesting
bid_date = '2026-06-04'
table_suffix = ''
repopulate_data = False


In [ ]:
portfolio = portfolio_normal_df.copy()
print(portfolio.shape)
portfolio.head()


In [ ]:
# portfolio_bq = bigquery_functions.upload_to_bq_from_dataframe(
#     portfolio,
#     dataset_name='constraint_family_exposure',
#     table_name='SPP_portfolio_to_cut',
#     temp=True
# )
# print('portfolio_bq:', portfolio_bq)

start_dt_cut, end_dt_cut = portfolio['dt'].min(), portfolio['dt'].max()
print(f'date range: {start_dt_cut} -> {end_dt_cut}')


In [ ]:
# run PortfolioExposureManager for SPP
portfolio_obj = PortfolioExposureManager(
    opexchange='SPP',
    portfolio_location=portfolio_bq,
    table_suffix=table_suffix,
    temp=False,
    start_dt=start_dt_cut,
    end_dt=end_dt_cut
)
print('PortfolioExposureManager initialized')

constraint_family_exposure_bq = portfolio_obj.get_portfolio_exposure_for_future_bids(
    mvalue_threshold=100,
    constraint_stats=True,
    methods_to_include=['dayzer_flow', 'constraint_feature', 'price_derived'],
    repopulate_data=repopulate_data
)
print('constraint_family_exposure_bq:', constraint_family_exposure_bq)


In [ ]:
# join exposure table with pre-calculated SPP quantiles
bq_query = f""" select a.dt, a.constraint_family_num, oops_constraint_num, monitored_clean, contingency_clean, KV, FlowRatio,
            rt_max_in7d, da_max_in7d, da_avg_in30d, short_bid_mw,
                adj_da_rt_norm_max_q95_0_98_1,
    adj_da_rt_norm_max_q95_0_95_0_98,
    adj_da_rt_norm_max_q95_0_75_0_95,
    adj_da_rt_norm_max_q95_0_0_75,
            from {constraint_family_exposure_bq} as a
            inner join `movetocloud-999.constraint_family_exposure.SPP_constraint_family_exposure_ve_prod_with_quantiles` as b
            on a.constraint_family_num = b.constraint_family_num and a.dt = b.dt
             where (da_max_in30d > 100 or rt_max_in30d > 100)  and short_bid_mw > 0"""

all_dt = bigquery_functions.download_df_from_bq(bq_query)
print(all_dt.shape)
all_dt.head()


In [ ]:
if all_dt.empty:
    print('all_dt is empty — no constraints above threshold, no scaling needed')
else:
    all_dt['constraint_family_num'] = all_dt['constraint_family_num'].astype(int)

    # attach ice price — use INDIANAHUB as the available ICE proxy for SPP
    ice_price_var = 'INDIANAHUB_ice_elec_price_forecast_f'
    ice_df, _ = ice_elec_price_vh.get_data_and_mapping_for_ice_elec(
        [636, ], 'SPP', ['INDIANAHUB'], start_dt_cut, end_dt_cut, var_spec=['f'], impute=True
    )
    ice_df = ice_df.rename(columns={ice_price_var: 'ice_price'})
    ice_df = ice_df.groupby('dt').agg({'ice_price': 'mean'}).reset_index()
    all_dt = pd.merge(all_dt, ice_df[['dt', 'ice_price']], how='left', on='dt')

    all_dt['FlowRatio'] = pd.to_numeric(all_dt['FlowRatio'], errors='coerce').fillna(0)
    all_dt['FlowRatio'] = all_dt['FlowRatio'].clip(upper=1)

    all_dt['KV'] = pd.to_numeric(all_dt['KV'], errors='coerce')
    all_dt['KV_group'] = pd.cut(
        all_dt['KV'],
        bins=[-float('inf'), 1, 115, 138, 345, float('inf')],
        labels=['01_na', '02_<=115', '03_138', '04_345', '05_>=500']
    ).astype(str)

    print(all_dt[['constraint_family_num', 'KV', 'KV_group', 'FlowRatio', 'ice_price']].head(10))


In [ ]:
scale = 3
base_limit = 1_000_000 * scale

risk_limit_rules = [
    ((all_dt['FlowRatio'] > 0.98) & (all_dt['KV_group'] == '02_<=115'), base_limit),
    ((all_dt['FlowRatio'] > 0.95) & (all_dt['FlowRatio'] <= 0.98) & (all_dt['KV_group'] == '02_<=115'), base_limit),
    ((all_dt['FlowRatio'] > 0.75) & (all_dt['FlowRatio'] <= 0.95) & (all_dt['KV_group'] == '02_<=115'), base_limit),
    ((all_dt['FlowRatio'] <= 0.75) & (all_dt['KV_group'] == '02_<=115'), base_limit),

    ((all_dt['FlowRatio'] > 0.98) & (all_dt['KV_group'] == '03_138'), base_limit),
    ((all_dt['FlowRatio'] > 0.95) & (all_dt['FlowRatio'] <= 0.98) & (all_dt['KV_group'] == '03_138'), base_limit),
    ((all_dt['FlowRatio'] > 0.75) & (all_dt['FlowRatio'] <= 0.95) & (all_dt['KV_group'] == '03_138'), base_limit),
    ((all_dt['FlowRatio'] <= 0.75) & (all_dt['KV_group'] == '03_138'), base_limit),

    ((all_dt['FlowRatio'] > 0.98) & (all_dt['KV_group'] == '04_345'), base_limit),
    ((all_dt['FlowRatio'] > 0.95) & (all_dt['FlowRatio'] <= 0.98) & (all_dt['KV_group'] == '04_345'), base_limit),
    ((all_dt['FlowRatio'] > 0.75) & (all_dt['FlowRatio'] <= 0.95) & (all_dt['KV_group'] == '04_345'), base_limit),
    ((all_dt['FlowRatio'] <= 0.75) & (all_dt['KV_group'] == '04_345'), base_limit),

    ((all_dt['FlowRatio'] > 0.98) & (all_dt['KV_group'] == '05_>=500'), base_limit),
    ((all_dt['FlowRatio'] > 0.95) & (all_dt['FlowRatio'] <= 0.98) & (all_dt['KV_group'] == '05_>=500'), base_limit),
    ((all_dt['FlowRatio'] > 0.75) & (all_dt['FlowRatio'] <= 0.95) & (all_dt['KV_group'] == '05_>=500'), base_limit),
    ((all_dt['FlowRatio'] <= 0.75) & (all_dt['KV_group'] == '05_>=500'), base_limit),
]

all_dt['risk_limit'] = np.select(
    [c for c, _ in risk_limit_rules],
    [v for _, v in risk_limit_rules],
    default=base_limit
)

all_dt['risk_limit'] = all_dt['risk_limit'] / all_dt['ice_price']

# KV-based multiplier
kv = pd.to_numeric(all_dt['KV'], errors='coerce')
kv_clean = kv.mask(kv <= 0)
all_dt['kv_factor'] = np.select(
    [kv_clean.le(69), kv_clean.le(115), kv_clean.le(161), kv_clean.le(220),
     kv_clean.le(345), kv_clean.le(500), kv_clean.gt(500)],
    [0.7, 1.1, 1.6, 2.1, 2.6, 3.1, 3.6],
    default=2.0
)
all_dt['risk_limit'] = all_dt['risk_limit'] * all_dt['kv_factor']

# sanitise column names for BigQuery upload
all_dt.columns = [re.sub(r'[^a-zA-Z0-9_]', '_', col) for col in all_dt.columns]

print(all_dt[['constraint_family_num', 'FlowRatio', 'KV_group', 'kv_factor', 'risk_limit']].head(10))


def _wind_scale_factor(row):
    """
    decide portfolio scale factor given daily wind percentile

    :author: jessica_xia
    :date: 2022-10-07
    :param row: 120DPec_daily_wind variable ranging from 0 to 1
    :return: int, wind scale factor
    """

    if row <= 0.05:
        return 0.4
    elif row <= 0.3:
        return 0.8
    elif row <= 0.9:
        return 1
    else:
        # 05/10/2022: change from 1.05 to 1.25
        return 1.25

def scale_by_wind(start_dt, end_dt, opexchange):


    wind_start_dt = (pd.to_datetime(start_dt) - pd.to_timedelta('125D')).strftime('%Y-%m-%d')
    wind_obj = wind.Wind(opexchange)
    daily_wind_df = wind_obj.get_total_wind(wind_start_dt, end_dt, var_spec=['f'], impute=True)

    # daily wind
    col = 'daily_total_wind_forecast_f'
    n_day = 120
    perc_col_name = str(n_day) + 'DPerc_' + col
    scale_col_name = 'scale_by_' + perc_col_name
    daily_wind_df = daily_wind_df.groupby(['dt'])['spp_wind_total_forecast_f'].agg(['mean']).reset_index()
    daily_wind_df.columns = ['dt', col]
    daily_wind_df[perc_col_name] = daily_wind_df[col].rolling(n_day, min_periods=1).apply(
        lambda x: pd.Series(x).rank(pct=True).iloc[-1])

    # create daily scale factor
    daily_wind_df[scale_col_name] = daily_wind_df[perc_col_name].apply(_wind_scale_factor)
#     daily_wind_df = daily_wind_df[['dt', scale_col_name]]
    daily_wind_df = daily_wind_df[(daily_wind_df['dt'] >= start_dt)&(daily_wind_df['dt'] <= end_dt)]

    return daily_wind_df

# darwin_wind_scale = scale_by_wind('2019-01-01', '2021-12-31', opexchange)
# display(darwin_wind_scale.head())
# table_fullname = "Darwin_SPP" + ".scaleFactorWind"
# sql_functions.replace_into_sql_table(darwin_wind_scale, table_fullname)

In [ ]:
risk_per_mw_rules = [
    ((all_dt['FlowRatio'] > 0.98) & (all_dt['KV_group'] == '02_<=115'),
     all_dt['adj_da_rt_norm_max_q95_0_98_1'].apply(lambda x: 46 if pd.isna(x) or x < 46 * 0.2 else x)),
    ((all_dt['FlowRatio'] > 0.95) & (all_dt['FlowRatio'] <= 0.98) & (all_dt['KV_group'] == '02_<=115'),
     all_dt['adj_da_rt_norm_max_q95_0_95_0_98'].fillna(26)),
    ((all_dt['FlowRatio'] > 0.75) & (all_dt['FlowRatio'] <= 0.95) & (all_dt['KV_group'] == '02_<=115'),
     all_dt['adj_da_rt_norm_max_q95_0_75_0_95'].fillna(20)),
    ((all_dt['FlowRatio'] <= 0.75) & (all_dt['KV_group'] == '02_<=115'),
     all_dt['adj_da_rt_norm_max_q95_0_0_75'].fillna(1)),

    ((all_dt['FlowRatio'] > 0.98) & (all_dt['KV_group'] == '03_138'),
     all_dt['adj_da_rt_norm_max_q95_0_98_1'].apply(lambda x: 42 if pd.isna(x) or x < 42 * 0.2 else x)),
    ((all_dt['FlowRatio'] > 0.95) & (all_dt['FlowRatio'] <= 0.98) & (all_dt['KV_group'] == '03_138'),
     all_dt['adj_da_rt_norm_max_q95_0_95_0_98'].fillna(28)),
    ((all_dt['FlowRatio'] > 0.75) & (all_dt['FlowRatio'] <= 0.95) & (all_dt['KV_group'] == '03_138'),
     all_dt['adj_da_rt_norm_max_q95_0_75_0_95'].fillna(20)),
    ((all_dt['FlowRatio'] <= 0.75) & (all_dt['KV_group'] == '03_138'),
     all_dt['adj_da_rt_norm_max_q95_0_0_75'].fillna(3)),

    ((all_dt['FlowRatio'] > 0.98) & (all_dt['KV_group'] == '04_345'),
     all_dt['adj_da_rt_norm_max_q95_0_98_1'].apply(lambda x: 35 if pd.isna(x) or x < 35 * 0.2 else x)),
    ((all_dt['FlowRatio'] > 0.95) & (all_dt['FlowRatio'] <= 0.98) & (all_dt['KV_group'] == '04_345'),
     all_dt['adj_da_rt_norm_max_q95_0_95_0_98'].fillna(21)),
    ((all_dt['FlowRatio'] > 0.75) & (all_dt['FlowRatio'] <= 0.95) & (all_dt['KV_group'] == '04_345'),
     all_dt['adj_da_rt_norm_max_q95_0_75_0_95'].fillna(17)),
    ((all_dt['FlowRatio'] <= 0.75) & (all_dt['KV_group'] == '04_345'),
     all_dt['adj_da_rt_norm_max_q95_0_0_75'].fillna(2)),

    ((all_dt['FlowRatio'] > 0.98) & (all_dt['KV_group'] == '05_>=500'),
     all_dt['adj_da_rt_norm_max_q95_0_98_1'].apply(lambda x: 43 if pd.isna(x) or x < 43 * 0.2 else x)),
    ((all_dt['FlowRatio'] > 0.95) & (all_dt['FlowRatio'] <= 0.98) & (all_dt['KV_group'] == '05_>=500'),
     all_dt['adj_da_rt_norm_max_q95_0_95_0_98'].fillna(28)),
    ((all_dt['FlowRatio'] > 0.75) & (all_dt['FlowRatio'] <= 0.95) & (all_dt['KV_group'] == '05_>=500'),
     all_dt['adj_da_rt_norm_max_q95_0_75_0_95'].fillna(24)),
    ((all_dt['FlowRatio'] <= 0.75) & (all_dt['KV_group'] == '05_>=500'),
     all_dt['adj_da_rt_norm_max_q95_0_0_75'].fillna(1)),
]

all_dt['risk_per_mw'] = np.select(
    [c for c, _ in risk_per_mw_rules],
    [v for _, v in risk_per_mw_rules],
    default=0.1
)

all_dt['risk_per_mw_recent'] = np.where(
    (all_dt['FlowRatio'] > 0.98) | (
        all_dt['FlowRatio'].isna() &
        (all_dt['rt_max_in7d'] > 10 * all_dt['da_avg_in30d']) &
        (all_dt['rt_max_in7d'] > 400)
    ),
    np.maximum(all_dt['rt_max_in7d'] / all_dt['ice_price'] - all_dt['da_max_in7d'] / all_dt['ice_price'], 0.1),
    0.1
)

all_dt['long_term_mw_limit'] = all_dt['risk_limit'] / all_dt['risk_per_mw']
all_dt['recent_mw_limit']    = all_dt['risk_limit'] / all_dt['risk_per_mw_recent']

base_limit = 1000 * scale
all_dt['KV_mw_limit'] = base_limit * all_dt['kv_factor']

print(all_dt[['constraint_family_num', 'risk_per_mw', 'risk_per_mw_recent',
              'long_term_mw_limit', 'recent_mw_limit', 'KV_mw_limit']].head(10))

target_scale_factor = {'Darwin': 1.0, 'Fourier': 1.0, 'Eigen': 1.0}
portfolio_precuts = get_portfolio(opexchange, start_dt, end_dt, strategy_list_dict, target_scale_factor)
# portfolio_precuts.to_csv(file_loc + 'portfolio_df_Nov2025.csv', index=False)




In [ ]:
# scale factors
all_dt['long_term_risk_scale_factor'] = 1.0
all_dt['recent_risk_scale_factor']    = 1.0
all_dt['kv_mw_scale_factor']          = 1.0

nonzero_mask = all_dt['short_bid_mw'] != 0

all_dt.loc[nonzero_mask, 'long_term_risk_scale_factor'] = (
    np.minimum(all_dt.loc[nonzero_mask, 'short_bid_mw'], all_dt.loc[nonzero_mask, 'long_term_mw_limit'])
    / all_dt.loc[nonzero_mask, 'short_bid_mw']
)
all_dt.loc[nonzero_mask, 'recent_risk_scale_factor'] = (
    np.minimum(all_dt.loc[nonzero_mask, 'short_bid_mw'], all_dt.loc[nonzero_mask, 'recent_mw_limit'])
    / all_dt.loc[nonzero_mask, 'short_bid_mw']
)
all_dt.loc[nonzero_mask, 'kv_mw_scale_factor'] = (
    np.minimum(all_dt.loc[nonzero_mask, 'short_bid_mw'], all_dt.loc[nonzero_mask, 'KV_mw_limit'])
    / all_dt.loc[nonzero_mask, 'short_bid_mw']
)

all_dt['final_factor'] = all_dt[
    ['long_term_risk_scale_factor', 'recent_risk_scale_factor', 'kv_mw_scale_factor']
].min(axis=1)

print(all_dt[['constraint_family_num', 'long_term_risk_scale_factor',
              'recent_risk_scale_factor', 'kv_mw_scale_factor', 'final_factor']].describe())


In [ ]:
# scale_table = bigquery_functions.upload_to_bq_from_dataframe(
#     all_dt[['dt', 'constraint_family_num', 'monitored_clean', 'KV_group', 'final_factor']],
#     'constraint_family_exposure',
#     'SPP_scale_table_temp',
#     temp=False
# )
# print('scale_table:', scale_table)


In [ ]:
# apply scale factors via BQ — join portfolio with SPP dfax and factor table
bq_query = f"""
WITH hr_table AS (
  SELECT x AS hr FROM UNNEST(GENERATE_ARRAY(1, 24)) AS x
),

factor_table AS (
  SELECT f.dt, f.node_num, h.hr,
         MIN(f.inc_factor) AS inc_factor,
         MIN(f.dec_factor) AS dec_factor
  FROM (
    SELECT dt, a.constraint_family_num, node_num, dfax, final_factor AS factor,
        CASE
          WHEN dfax > 0 AND KV_group IN (\"02_<=115\", \"03_138\", \"04_345\", \"05_>=500\") THEN final_factor
          ELSE 1
        END AS inc_factor,
        CASE
          WHEN dfax < 0 AND KV_group IN (\"02_<=115\", \"03_138\", \"04_345\", \"05_>=500\") THEN final_factor
          ELSE 1
        END AS dec_factor
    FROM `movetocloud-999.{scale_table}` AS a
    LEFT JOIN `movetocloud-999.constraint_family_exposure.SPP_dfax_ve_prod` AS b
      ON a.constraint_family_num = b.constraintFamilyNum
    WHERE ABS(dfax) > 0.05
  ) AS f
  CROSS JOIN hr_table AS h
  GROUP BY f.dt, f.node_num, h.hr
)

SELECT
  M.* EXCEPT(dt),
  CAST(M.dt AS STRING) AS dt,
  CASE
    WHEN M.incdec = 'Increment' THEN COALESCE(M.bid_mw * N.inc_factor, M.bid_mw)
    ELSE COALESCE(M.bid_mw * N.dec_factor, M.bid_mw)
  END AS scaled_bid_mw

FROM `{portfolio_bq}` AS M

INNER JOIN (
  SELECT DISTINCT dt FROM `movetocloud-999.{scale_table}`
) AS Q ON CAST(M.dt AS STRING) = Q.dt

LEFT JOIN factor_table AS N
  ON CAST(M.dt AS STRING) = N.dt AND M.hr = N.hr AND M.node_num = N.node_num
"""

portfolio_scaled = bigquery_functions.download_df_from_bq(bq_query)
portfolio_scaled['bid_mw_original'] = portfolio_scaled['bid_mw']
portfolio_scaled['bid_mw'] = portfolio_scaled['scaled_bid_mw']

print(portfolio_scaled.shape)
portfolio_scaled.head()


In [ ]:
# inspect bid_mw reduction
diff = portfolio_scaled.groupby('incdec').agg(
    bid_mw_original=('bid_mw_original', 'sum'),
    bid_mw_scaled=('bid_mw', 'sum')
).reset_index()
diff['reduction_pct'] = (1 - diff['bid_mw_scaled'] / diff['bid_mw_original']) * 100
print(diff)

# nodes most affected
node_diff = portfolio_scaled.groupby(['node_num', 'incdec']).agg(
    bid_mw_original=('bid_mw_original', 'sum'),
    bid_mw_scaled=('bid_mw', 'sum')
).reset_index()
node_diff['reduction'] = node_diff['bid_mw_original'] - node_diff['bid_mw_scaled']
node_diff.sort_values('reduction', ascending=False).head(20)
